## Step 1 - Convert City Name to Coordinates
This cell calls the Open‑Meteo Geocoding API to convert a city name into geographic coordinates (latitude and longitude). Weather APIs typically require coordinates instead of a city name.

In [ ]:
import requests
#import openmeteo_requests
import json

## Inspect the Geocoding JSON Response
Here we print the JSON response from the geocoding API to understand the structure of the returned data.

In [18]:
url = "https://geocoding-api.open-meteo.com/v1/search"
params = {
    "name": "Riga",
    "count": 1,
    "language": "en",
    "format": "json"
}

response = requests.get(url, params=params)
json_response = response.json()

print(f"URL: {url}")
print(f"Status Code: {response.status_code}")
#print(json_response)
print(f"JSON Response:\n {json.dumps(json_response, indent=4, separators=(", ", ": "))}")

URL: https://geocoding-api.open-meteo.com/v1/search
Status Code: 200
JSON Response:
 {
    "results": [
        {
            "id": 456172, 
            "name": "Riga", 
            "latitude": 56.946, 
            "longitude": 24.10589, 
            "elevation": 6.0, 
            "feature_code": "PPLC", 
            "country_code": "LV", 
            "admin1_id": 456173, 
            "admin2_id": 11352868, 
            "timezone": "Europe/Riga", 
            "population": 742572, 
            "country_id": 458258, 
            "country": "Latvia", 
            "admin1": "R\u012bga", 
            "admin2": "R\u012bga"
        }
    ], 
    "generationtime_ms": 0.8056164
}


## Extract Relevant Fields
This cell extracts useful information such as city name, country, latitude, and longitude from the JSON response.

In [23]:
city_data = json_response["results"][0]

city = city_data["name"]
country = city_data["country"]
latitude = city_data["latitude"]
longitude = city_data["longitude"]

print(city, country, latitude, longitude)

Riga Latvia 56.946 24.10589


## Step 2 - Request Weather Data
Using the coordinates obtained earlier, this cell calls the Open‑Meteo Forecast API to retrieve current weather information.

In [28]:
forecast_url ="https://api.open-meteo.com/v1/forecast"
forecast_params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": "temperature_2m,wind_speed_10m"
}

weather_response = requests.get(forecast_url, params=forecast_params)
weather_json = weather_response.json()

print(weather_json)

print(f"JSON Response:\n {json.dumps(weather_json, indent=4, separators=(", ", ": "))}")

{'latitude': 56.944096, 'longitude': 24.109085, 'generationtime_ms': 0.02765655517578125, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 6.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-04-11T17:30', 'interval': 900, 'temperature_2m': 8.0, 'wind_speed_10m': 12.6}}
JSON Response:
 {
    "latitude": 56.944096, 
    "longitude": 24.109085, 
    "generationtime_ms": 0.02765655517578125, 
    "utc_offset_seconds": 0, 
    "timezone": "GMT", 
    "timezone_abbreviation": "GMT", 
    "elevation": 6.0, 
    "current_units": {
        "time": "iso8601", 
        "interval": "seconds", 
        "temperature_2m": "\u00b0C", 
        "wind_speed_10m": "km/h"
    }, 
    "current": {
        "time": "2026-04-11T17:30", 
        "interval": 900, 
        "temperature_2m": 8.0, 
        "wind_speed_10m": 12.6
    }
}


## Inspect Weather JSON
Print the weather API response to see where temperature, wind speed, and time are located in the JSON structure.

In [29]:
current = weather_json["current"]

temperature = current["temperature_2m"]
wind_speed = current["wind_speed_10m"]

print("Temperature:", temperature)
print("Wind speed:", wind_speed)

Temperature: 8.0
Wind speed: 12.6


## Extract Weather Values
Here we extract the temperature, wind speed, and timestamp from the weather API response.

In [32]:
print("City:", city)
print("Country:", country)
print("Latitude:", latitude)
print("Longitude:", longitude)

print("Temperature:", temperature)
print("Wind speed:", wind_speed)

City: Riga
Country: Latvia
Latitude: 56.946
Longitude: 24.10589
Temperature: 8.0
Wind speed: 12.6


## Create Reusable Function
Finally, we wrap the logic into a function that takes a city name as input and returns weather information.

In [ ]:
def get_weather_by_city(city):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json"
    }
    response = requests.get(url, params=params)
    json_response = response.json()
    city_data = json_response["results"][0]
    latitude = city_data["latitude"]
    longitude = city_data["longitude"]

    forecast_url ="https://api.open-meteo.com/v1/forecast"
    forecast_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,wind_speed_10m"
    }
    weather_response = requests.get(forecast_url, params=forecast_params)
    weather_json = weather_response.json()
    current = weather_json["current"]
    temperature = current["temperature_2m"]
    wind_speed = current["wind_speed_10m"]

    if "results" not in json_response:
        return {"error": "City not found"}

    return {
        "name": city,
        "temperature": temperature,
        "wind_speed": wind_speed
    }